# 106 - Live Option Chain (In-Notebook)

A live-updating option chain rendered inside the notebook with
`ipywidgets`. It reads the current spot and snapshot quotes for SPY, pulls
Greeks straight from the snapshot endpoint, and renders a color-coded
chain: calls on the left, strike in the center, puts on the right.

Snapshot data refreshes through the trading day; quote and Greeks
snapshots return no data when the market is closed.

Install `ipywidgets` if you do not have it: `pip install ipywidgets`.

In [ ]:
from thetadatadx import Credentials, Config, Client
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import time

creds = Credentials.from_file("creds.txt")
tdx = Client(creds, Config.production())
print("Connected:", tdx)

## Choose Ticker and Expiration

Expirations are ISO date strings; we keep upcoming ones and build a
human-readable dropdown.

In [ ]:
ticker = "SPY"

# Spot from a stock OHLC snapshot (one row per symbol).
snap = tdx.historical.stock_snapshot_ohlc([ticker])
spot = snap[0].close if snap else 0.0
print(f"{ticker} spot: ${spot:.2f}")

# Upcoming expirations
expirations = tdx.historical.option_list_expirations(ticker).to_list()
today = datetime.now()
expirations = [e for e in expirations if e > today.date().isoformat()]
print(f"{ticker}: {len(expirations)} upcoming expirations")

# Readable labels for the nearest 20
exp_labels = []
for exp_str in expirations[:20]:
    exp_dt = datetime.strptime(exp_str, "%Y-%m-%d")
    dte = (exp_dt - today).days
    exp_labels.append(f"{exp_str}  ({exp_dt.strftime('%b %d')} - {dte}d)")

exp_dropdown = widgets.Dropdown(
    options=list(zip(exp_labels, expirations[:20])),
    description="Expiration:",
    style={"description_width": "initial"},
)
display(exp_dropdown)

## Build the Option Chain

For each strike near ATM, fetch the call and put quote, last trade, open
interest, and snapshot Greeks. Snapshot methods are keyword-only for
`expiration` / `strike` / `right`, take the strike as a dollar string, and
return a list of typed rows.

In [ ]:
def build_option_chain(ticker, expiration, spot_price, num_strikes=20):
    # Fetch quotes, trades, OI, and snapshot Greeks; return a DataFrame.
    all_strikes = tdx.historical.option_list_strikes(ticker, expiration).to_list()
    if not all_strikes:
        return pd.DataFrame()

    # Strikes are dollar strings; sort by value and window around ATM.
    strike_pairs = sorted(((s, float(s)) for s in all_strikes), key=lambda x: x[1])
    atm_idx = min(range(len(strike_pairs)),
                  key=lambda i: abs(strike_pairs[i][1] - spot_price))
    half = num_strikes // 2
    selected = strike_pairs[max(0, atm_idx - half):atm_idx + half + 1]

    rows = []
    for strike_str, strike_val in selected:
        row = {"strike": strike_val}

        for right, prefix in [("C", "call_"), ("P", "put_")]:
            # Quote (bid / ask)
            try:
                quote = tdx.historical.option_snapshot_quote(
                    ticker, expiration=expiration, strike=strike_str, right=right)
            except Exception:
                quote = None
            if quote:
                q = quote[0]
                row[f"{prefix}bid"], row[f"{prefix}ask"] = q.bid, q.ask
            else:
                row[f"{prefix}bid"] = row[f"{prefix}ask"] = np.nan

            # Last trade
            try:
                trade = tdx.historical.option_snapshot_trade(
                    ticker, expiration=expiration, strike=strike_str, right=right)
            except Exception:
                trade = None
            if trade:
                row[f"{prefix}last"] = trade[0].price
                row[f"{prefix}volume"] = trade[0].size
            else:
                row[f"{prefix}last"], row[f"{prefix}volume"] = np.nan, 0

            # Open interest (updated overnight)
            try:
                oi = tdx.historical.option_snapshot_open_interest(
                    ticker, expiration=expiration, strike=strike_str, right=right)
                row[f"{prefix}oi"] = oi[0].open_interest if oi else 0
            except Exception:
                row[f"{prefix}oi"] = 0

            # Snapshot Greeks (computed server-side)
            try:
                g = tdx.historical.option_snapshot_greeks_all(
                    ticker, expiration=expiration, strike=strike_str, right=right)
            except Exception:
                g = None
            if g and g[0].implied_volatility and g[0].implied_volatility > 0:
                gr = g[0]
                row[f"{prefix}iv"] = gr.implied_volatility
                row[f"{prefix}delta"] = gr.delta
                row[f"{prefix}gamma"] = gr.gamma
                row[f"{prefix}theta"] = gr.theta
            else:
                for k in ("iv", "delta", "gamma", "theta"):
                    row[f"{prefix}{k}"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows).sort_values("strike").reset_index(drop=True)


def style_chain(df, spot_price):
    # Option-chain styling: ITM tint, ATM highlight, formatted numbers.
    call_cols = ["call_iv", "call_delta", "call_gamma", "call_theta",
                 "call_bid", "call_ask", "call_last", "call_volume", "call_oi"]
    put_cols = ["put_bid", "put_ask", "put_last", "put_volume", "put_oi",
                "put_iv", "put_delta", "put_gamma", "put_theta"]
    display_cols = [c for c in call_cols if c in df.columns]
    display_cols.append("strike")
    display_cols.extend([c for c in put_cols if c in df.columns])

    styled_df = df[display_cols].copy()

    rename = {}
    for c in styled_df.columns:
        parts = c.split("_", 1)
        if len(parts) == 2 and parts[0] in ("call", "put"):
            rename[c] = f"{parts[0][0].upper()}_{parts[1].upper()}"
        else:
            rename[c] = c.upper()
    styled_df.columns = [rename.get(c, c) for c in styled_df.columns]

    atm_strike = df.loc[(df["strike"] - spot_price).abs().idxmin(), "strike"]

    def highlight_itm(row):
        styles = [""] * len(row)
        if "STRIKE" not in row.index:
            return styles
        strike = row["STRIKE"]
        strike_pos = list(row.index).index("STRIKE")
        if strike < spot_price:  # ITM calls
            for i in range(strike_pos):
                styles[i] = "background-color: rgba(76, 175, 80, 0.15)"
        if strike > spot_price:  # ITM puts
            for i in range(strike_pos + 1, len(row)):
                styles[i] = "background-color: rgba(76, 175, 80, 0.15)"
        if strike == atm_strike:
            styles[strike_pos] = (
                "background-color: rgba(255, 215, 0, 0.35); font-weight: bold")
        return styles

    fmt = {}
    for c in styled_df.columns:
        if "IV" in c:
            fmt[c] = lambda x: f"{x*100:.1f}%" if pd.notna(x) else ""
        elif any(g in c for g in ("DELTA", "GAMMA", "THETA")):
            fmt[c] = lambda x: f"{x:.4f}" if pd.notna(x) else ""
        elif "VOLUME" in c or "OI" in c:
            fmt[c] = lambda x: f"{int(x):,}" if pd.notna(x) and x != 0 else ""
        elif c == "STRIKE":
            fmt[c] = lambda x: f"${x:.1f}" if pd.notna(x) else ""
        else:
            fmt[c] = lambda x: f"{x:.2f}" if pd.notna(x) else ""

    return styled_df.style.apply(highlight_itm, axis=1).format(fmt)


print("build_option_chain() and style_chain() defined.")

In [ ]:
chain = build_option_chain(ticker, exp_dropdown.value, spot)
exp_dt = datetime.strptime(exp_dropdown.value, "%Y-%m-%d")
dte = (exp_dt - datetime.now()).days

display(HTML(
    f"<h3>{ticker} Option Chain &mdash; {exp_dropdown.value} "
    f"({exp_dt.strftime('%b %d, %Y')}, {dte} DTE) &mdash; "
    f"Spot ${spot:.2f}</h3>"
))
display(style_chain(chain, spot))

## Live Auto-Refresh

A Refresh button plus an auto-refresh toggle. Re-fetches the spot and
rebuilds the chain on each pass.

In [ ]:
output = widgets.Output()
refresh_btn = widgets.Button(description="Refresh", button_style="info", icon="refresh")
auto_toggle = widgets.ToggleButton(value=False, description="Auto-refresh (5s)", icon="repeat")
strikes_slider = widgets.IntSlider(value=20, min=5, max=50, step=5, description="Strikes:")
controls = widgets.HBox([refresh_btn, auto_toggle, strikes_slider])


def refresh(_=None):
    with output:
        clear_output(wait=True)

        snap = tdx.historical.stock_snapshot_ohlc([ticker])
        current_spot = snap[0].close if snap else spot

        chain = build_option_chain(
            ticker, exp_dropdown.value, current_spot,
            num_strikes=strikes_slider.value,
        )
        if chain.empty:
            print("No data returned (market may be closed).")
            return

        exp_dt = datetime.strptime(exp_dropdown.value, "%Y-%m-%d")
        dte = (exp_dt - datetime.now()).days
        ts = time.strftime("%H:%M:%S")

        display(HTML(
            f"<div style='display:flex; justify-content:space-between; align-items:baseline;'>"
            f"<span style='color:#4FC3F7; font-weight:bold; font-size:14px;'>CALLS</span>"
            f"<span style='font-size:13px;'>{ticker} &mdash; {exp_dropdown.value} "
            f"({dte} DTE) &mdash; Spot ${current_spot:.2f} &mdash; {ts}</span>"
            f"<span style='color:#FF8A65; font-weight:bold; font-size:14px;'>PUTS</span>"
            f"</div>"
        ))
        display(style_chain(chain, current_spot))


refresh_btn.on_click(refresh)
display(controls, output)
refresh()

In [ ]:
# Run this cell to start auto-refresh. Interrupt the kernel to stop.
while auto_toggle.value:
    refresh()
    time.sleep(5)

## IV Smile Visualization

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

chain = build_option_chain(ticker, exp_dropdown.value, spot, num_strikes=30)

fig, ax = plt.subplots(figsize=(12, 5))

valid_call = chain.dropna(subset=["call_iv"])
valid_put = chain.dropna(subset=["put_iv"])

if len(valid_call) > 0:
    ax.plot(valid_call["strike"], valid_call["call_iv"] * 100,
            "o-", color="#1f77b4", markersize=4, linewidth=1.2, label="Call IV")
if len(valid_put) > 0:
    ax.plot(valid_put["strike"], valid_put["put_iv"] * 100,
            "s-", color="#d62728", markersize=4, linewidth=1.2, label="Put IV")

ax.axvline(x=spot, color="gray", linestyle="--", linewidth=1, alpha=0.7,
           label=f"Spot ${spot:.0f}")
ax.set_xlabel("Strike ($)")
ax.set_ylabel("Implied Volatility (%)")

exp_dt = datetime.strptime(exp_dropdown.value, "%Y-%m-%d")
dte = (exp_dt - datetime.now()).days
ax.set_title(f"{ticker} IV Smile - {exp_dropdown.value} ({dte} DTE)")
ax.legend()
plt.tight_layout()
plt.show()

## Multi-Expiration Term Structure

In [ ]:
atm_ivs = []
for exp_str in expirations[:10]:
    try:
        ch = build_option_chain(ticker, exp_str, spot, num_strikes=3)
        if ch.empty or "call_iv" not in ch.columns:
            continue
        atm_row = ch.loc[(ch["strike"] - spot).abs().idxmin()]
        call_iv = atm_row.get("call_iv", np.nan)
        put_iv = atm_row.get("put_iv", np.nan)
        if pd.notna(call_iv) and call_iv > 0:
            dte = (datetime.strptime(exp_str, "%Y-%m-%d") - datetime.now()).days
            atm_ivs.append({
                "Expiration": exp_str,
                "DTE": dte,
                "Call ATM IV": call_iv,
                "Put ATM IV": put_iv if pd.notna(put_iv) else np.nan,
            })
            print(f"  {exp_str} ({dte:>3}d): Call IV {call_iv*100:.1f}%")
    except Exception as exc:
        print(f"  {exp_str}: skipped ({exc})")
        continue

if atm_ivs:
    term = pd.DataFrame(atm_ivs)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(term["DTE"], term["Call ATM IV"] * 100, "o-",
            color="#1f77b4", linewidth=2, label="Call ATM IV")
    if term["Put ATM IV"].notna().any():
        ax.plot(term["DTE"], term["Put ATM IV"] * 100, "s-",
                color="#d62728", linewidth=2, label="Put ATM IV")
    ax.set_xlabel("Days to Expiration")
    ax.set_ylabel("ATM Implied Volatility (%)")
    ax.set_title(f"{ticker} IV Term Structure")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No ATM IV data collected.")

---

**Previous:** [105 - Real-Time Streaming](105_realtime_streaming.ipynb)
**Next:** [107 - Full Trade Stream](107_full_trade_stream.ipynb)